# 🚀 First Initial Analysis

In [2]:
import os
import sys
from pathlib import Path

# If notebook is inside <repo>/notebooks/, this moves one level up to repo root
PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current working dir:", Path.cwd().resolve())
print("Project root added to sys.path:", PROJECT_ROOT)
print("cf_copilot exists:", (PROJECT_ROOT / "cf_copilot").exists())


Current working dir: /home/saiyudh/code/EwaltsJ/cf_copilot/notebooks
Project root added to sys.path: /home/saiyudh/code/EwaltsJ/cf_copilot
cf_copilot exists: True


In [4]:
import pandas as pd
import numpy as np

from cf_copilot.ml_logic.data import load_cashflow_data, data_cleaning, build_sliding_window_snapshots
from cf_copilot.ml_logic.encoders import preprocess

pd.set_option("display.max_columns", 200)

# 1) Build the same training table as your pipeline
df_raw = load_cashflow_data()
df_clean = data_cleaning(df_raw)
big_df = build_sliding_window_snapshots(df_clean)

# 2) Same time split logic
big_df = big_df.sort_values("invoice_sent").reset_index(drop=True)
cutoff_date = big_df["invoice_sent"].quantile(0.8)

train_df = big_df[big_df["reference_date"] <= cutoff_date].copy()
test_df  = big_df[big_df["reference_date"] > cutoff_date].copy()

print("big_df shape:", big_df.shape)
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

print("\nWeek bucket distribution - full")
print(big_df["week_bucket"].value_counts(normalize=True).sort_index().round(4))

print("\nWeek bucket distribution - train")
print(train_df["week_bucket"].value_counts(normalize=True).sort_index().round(4))

print("\nWeek bucket distribution - test")
print(test_df["week_bucket"].value_counts(normalize=True).sort_index().round(4))

# 3) How often the same invoice is repeated after augmentation
dup_counts = big_df.groupby("doc_id").size().describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
print("\nSnapshot repetition per invoice")
print(dup_counts)

# 4) Relation between repetition count and target
doc_repeat = big_df.groupby("doc_id").size().rename("n_snapshots")
doc_bucket = big_df.groupby("doc_id")["week_bucket"].max().rename("max_week_bucket")
repeat_vs_bucket = pd.concat([doc_repeat, doc_bucket], axis=1)

print("\nMean snapshot count by max_week_bucket")
print(repeat_vs_bucket.groupby("max_week_bucket")["n_snapshots"].mean().round(2))

# 5) Numeric columns to inspect
numeric_cols = [
    "invoice_age_days", "days_until_due", "pay_terms_days", "days_past_due",
    "customer_avg_delay", "late_payment_ratio", "prev_transaction_count",
    "days_since_last_invoice", "customer_risk_score", "invoice_amount",
    "invoice_amount_log"
]

summary = train_df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
summary["iqr"] = summary["75%"] - summary["25%"]
summary["upper_iqr_bound"] = summary["75%"] + 1.5 * summary["iqr"]
summary["lower_iqr_bound"] = summary["25%"] - 1.5 * summary["iqr"]

def outlier_rate_iqr(s):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    return ((s < lo) | (s > hi)).mean()

summary["iqr_outlier_rate"] = [outlier_rate_iqr(train_df[c].dropna()) for c in numeric_cols]
summary = summary.sort_values("iqr_outlier_rate", ascending=False)

print("\nNumeric feature outlier summary")
display(summary)


Loading local file: /home/saiyudh/code/EwaltsJ/cf_copilot/raw_data/dataset.csv
Original rows: 39152
Augmented rows: 98169
week_bucket
1    38825
2    33060
3    10401
4     4009
5     3172
6     2192
7     6510
Name: count, dtype: int64
big_df shape: (98169, 29)
train_df shape: (76358, 29)
test_df shape: (21811, 29)

Week bucket distribution - full
week_bucket
1    0.3955
2    0.3368
3    0.1059
4    0.0408
5    0.0323
6    0.0223
7    0.0663
Name: proportion, dtype: float64

Week bucket distribution - train
week_bucket
1    0.3821
2    0.3337
3    0.1078
4    0.0422
5    0.0343
6    0.0246
7    0.0752
Name: proportion, dtype: float64

Week bucket distribution - test
week_bucket
1    0.4423
2    0.3474
3    0.0995
4    0.0359
5    0.0254
6    0.0143
7    0.0352
Name: proportion, dtype: float64

Snapshot repetition per invoice
count    38901.000000
mean         2.523560
std          1.889109
min          1.000000
50%          2.000000
75%          3.000000
90%          4.000000
95%     

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,iqr,upper_iqr_bound,lower_iqr_bound,iqr_outlier_rate
pay_terms_days,76358.0,22.522539,18.348700,0.000000,0.000000,10.000000,15.000000,15.000000,20.000000,65.000000,90.000000,120.000000,5.000000,27.500000,7.500000,0.259881
days_until_due,76358.0,9.572396,18.211193,-197.000000,-55.000000,-15.000000,4.000000,9.000000,14.000000,39.000000,64.430000,120.000000,10.000000,29.000000,-11.000000,0.138964
days_past_due,76358.0,-9.572396,18.211193,-120.000000,-64.430000,-39.000000,-14.000000,-9.000000,-4.000000,15.000000,55.000000,197.000000,10.000000,11.000000,-29.000000,0.138964
prev_transaction_count,76358.0,604.373242,1339.449579,0.000000,1.000000,2.000000,17.000000,85.000000,348.000000,4361.000000,6128.000000,6462.000000,331.000000,844.500000,-479.500000,0.138492
days_since_last_invoice,76130.0,4.488914,9.586513,1.000000,1.000000,1.000000,1.000000,1.000000,4.000000,16.000000,46.000000,244.000000,3.000000,8.500000,-3.500000,0.134927
customer_avg_delay,70035.0,4.155802,9.813934,0.000000,0.000000,0.000000,0.599678,1.108108,2.672414,24.500000,46.333333,120.000000,2.072736,5.781518,-2.509426,0.131177
customer_risk_score,76358.0,1.449390,2.962734,0.000000,0.000000,0.000000,0.257944,0.585714,1.212903,5.700000,13.759206,36.700000,0.954959,2.645342,-1.174495,0.108895
invoice_age_days,76358.0,12.950143,16.922679,0.000000,0.000000,0.000000,3.000000,8.000000,14.000000,49.000000,84.000000,199.000000,11.000000,30.500000,-13.500000,0.106773
invoice_amount,76358.0,28645.142274,35321.898023,2.400000,58.410000,300.623000,3803.380000,15601.770000,41981.017500,100712.158500,152612.550000,632134.240000,38177.637500,99247.473750,-53463.076250,0.052241
invoice_amount_log,76358.0,9.262288,1.805278,1.223775,4.084463,5.709178,8.243908,9.655204,10.644997,11.520032,11.935664,13.356859,2.401088,14.246629,4.642276,0.017366


In [5]:
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

audit_numeric_cols = [
    "business_year", "invoice_age_days", "days_until_due", "pay_terms_days",
    "invoice_month", "due_month", "days_past_due", "customer_avg_delay",
    "late_payment_ratio", "prev_transaction_count", "days_since_last_invoice",
    "customer_risk_score", "invoice_amount", "invoice_amount_log"
]

iso_input = train_df[audit_numeric_cols].copy()

iso = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", IsolationForest(
        n_estimators=300,
        contamination=0.01,   # start conservative
        random_state=42,
        n_jobs=-1
    ))
])

iso.fit(iso_input)

train_df["iso_pred"] = iso.named_steps["model"].fit_predict(
    SimpleImputer(strategy="median").fit_transform(iso_input)
)
train_df["is_outlier_iso"] = (train_df["iso_pred"] == -1).astype(int)

print("Outlier rate:", train_df["is_outlier_iso"].mean().round(4))

outlier_profile = (
    train_df.groupby("is_outlier_iso")[audit_numeric_cols + ["week_bucket"]]
    .median(numeric_only=True)
    .T
)

display(outlier_profile)

suspects = train_df.loc[train_df["is_outlier_iso"] == 1, [
    "doc_id", "cust_number", "reference_date", "week_bucket",
    "invoice_amount", "invoice_age_days", "days_until_due",
    "days_past_due", "customer_avg_delay", "late_payment_ratio",
    "prev_transaction_count", "days_since_last_invoice"
]].sort_values(
    ["invoice_amount", "invoice_age_days", "days_past_due"],
    ascending=[False, False, False]
)

display(suspects.head(30))


Outlier rate: 0.01


is_outlier_iso,0,1
business_year,2019.000000,2019.000000
invoice_age_days,8.000000,72.000000
days_until_due,9.000000,-53.000000
pay_terms_days,15.000000,10.000000
invoice_month,6.000000,5.000000
due_month,6.000000,6.000000
days_past_due,-9.000000,53.000000
customer_avg_delay,1.096257,50.800000
late_payment_ratio,0.429878,0.946429
prev_transaction_count,86.000000,19.500000


,doc_id,cust_number,reference_date,week_bucket,invoice_amount,invoice_age_days,days_until_due,days_past_due,customer_avg_delay,late_payment_ratio,prev_transaction_count,days_since_last_invoice
45308,1929556008,0100031723,2019-06-28,4,227086.53000,0.0,30.0,-30,0.000000,0.000000,2,77.0
2196,1928608865,0200029010,2019-03-01,1,200661.87000,49.0,-34.0,34,0.000000,0.000000,2,49.0
2312,1928608865,0200029010,2019-02-22,2,200661.87000,42.0,-27.0,27,0.000000,0.000000,2,42.0
22347,1929040321,0200740737,2019-06-21,1,163871.43000,84.0,-44.0,44,NaN,NaN,2,57.0
22335,1929040321,0200740737,2019-06-14,2,163871.43000,77.0,-37.0,37,NaN,NaN,2,50.0
71863,1930070261,CCU013,2019-11-08,3,134242.20000,27.0,-27.0,27,43.158904,1.000000,430,2.0
71870,1930070261,CCU013,2019-11-01,4,134242.20000,20.0,-20.0,20,43.158904,1.000000,421,1.0
59383,1929774239,CCU013,2019-09-20,1,132317.70000,27.0,-27.0,27,43.530686,1.000000,348,2.0
37022,1991827183,CC6050,2019-08-09,1,127438.00000,76.0,-31.0,31,32.500000,1.000000,14,24.0
37019,1991827183,CC6050,2019-08-02,2,127438.00000,69.0,-24.0,24,32.500000,1.000000,14,17.0


In [6]:
def cap_iqr(df, cols, factor=1.5):
    df = df.copy()
    bounds = {}
    for c in cols:
        q1 = df[c].quantile(0.25)
        q3 = df[c].quantile(0.75)
        iqr = q3 - q1
        lo = q1 - factor * iqr
        hi = q3 + factor * iqr
        df[c] = df[c].clip(lower=lo, upper=hi)
        bounds[c] = {"lower": lo, "upper": hi}
    return df, bounds

cap_cols = [
    "invoice_amount",
    "customer_avg_delay",
    "prev_transaction_count",
    "days_since_last_invoice",
    "days_past_due",
    "invoice_age_days"
]

train_capped, cap_bounds = cap_iqr(train_df, cap_cols, factor=1.5)
test_capped = test_df.copy()

for c in cap_cols:
    test_capped[c] = test_capped[c].clip(
        lower=cap_bounds[c]["lower"],
        upper=cap_bounds[c]["upper"]
    )

print(cap_bounds)


{'invoice_amount': {'lower': -53463.076250000006, 'upper': 99247.47375}, 'customer_avg_delay': {'lower': -2.5094261669004405, 'upper': 5.781517769105782}, 'prev_transaction_count': {'lower': -479.5, 'upper': 844.5}, 'days_since_last_invoice': {'lower': -3.5, 'upper': 8.5}, 'days_past_due': {'lower': -29.0, 'upper': 11.0}, 'invoice_age_days': {'lower': -13.5, 'upper': 30.5}}


# 🚀 Finding After Initial Analysis
issue looks less like “bad outliers” and more like augmentation bias plus heavy-tailed business features. Isolation Forest is useful here as a ranking tool for unusual rows, but the rows it flagged are mostly old, overdue, high-delay invoices rather than obvious garbage records, which means we should be careful not to delete them blindly. Also, polynomial features should still stay on the back burner for now because tree models already capture nonlinear splits, so the bigger win is likely fixing duplicated-snapshot weighting first.

🔥 What stands out
A few important things from your output:

1) week_bucket is strongly front-loaded, but that is expected given your weekly snapshot design.
2) The same invoice is being repeated across snapshots, and longer-to-pay invoices appear more often.

💡 The outlier profile is dominated by:

1) very high invoice_age_days
2) strongly negative days_until_due
3) high days_past_due
4) very high customer_avg_delay
5) very high late_payment_ratio

invoice_amount is not the main abnormality by itself, because median amount for outliers vs non-outliers is quite similar in your Isolation Forest profile.

👍 Most important practical takeaway: do not use that IQR capping block as-is. In your data, it is too aggressive for valid business features like customer_avg_delay, days_since_last_invoice, and days_past_due.

✅ Decision for next step
test sample weighting by invoice repetition before touching polynomial features or hard outlier removal.

Why:

a) repeated snapshots for the same doc_id are likely amplifying long-payment invoices

b) this can skew the model toward learning “how long an invoice stays around in the augmented table” rather than just “when it will get paid”

c) RandomForest supports sample_weight during fitting, so this is an easy and clean test

# 🚀 Analaysis Round 2

 ✅ What I want now:

1) sample_weight.describe()
2) mean snapshot count by week_bucket
3) weighted class mass by week_bucket

In [7]:
# Add per-doc repetition counts on the augmented data
doc_counts_train = train_df.groupby("doc_id").size().rename("doc_snapshot_count")
doc_counts_test = test_df.groupby("doc_id").size().rename("doc_snapshot_count")

train_df_w = train_df.merge(doc_counts_train, on="doc_id", how="left")
test_df_w = test_df.merge(doc_counts_test, on="doc_id", how="left")

# Weight each repeated invoice snapshot inversely to how often it appears
train_df_w["sample_weight"] = 1.0 / train_df_w["doc_snapshot_count"]
test_df_w["sample_weight"] = 1.0 / test_df_w["doc_snapshot_count"]

print(train_df_w[["doc_id", "doc_snapshot_count", "sample_weight"]].head())
print("\nTrain weight summary:")
print(train_df_w["sample_weight"].describe())

print("\nMean snapshot count by week bucket (train):")
print(train_df_w.groupby("week_bucket")["doc_snapshot_count"].mean().round(3))

print("\nWeighted class mass by week bucket (train):")
weighted_bucket_mass = train_df_w.groupby("week_bucket")["sample_weight"].sum()
weighted_bucket_mass = weighted_bucket_mass / weighted_bucket_mass.sum()
print(weighted_bucket_mass.round(4))


       doc_id  doc_snapshot_count  sample_weight
0  1928721098                  10       0.100000
1  1928721098                  10       0.100000
2  1928721098                  10       0.100000
3  1928721098                  10       0.100000
4  2960521184                   6       0.166667

Train weight summary:
count    76358.000000
mean         0.396160
std          0.219942
min          0.034483
25%          0.200000
50%          0.500000
75%          0.500000
max          1.000000
Name: sample_weight, dtype: float64

Mean snapshot count by week bucket (train):
week_bucket
1     2.520
2     2.762
3     4.456
4     6.844
5     7.549
6     8.605
7    11.985
Name: doc_snapshot_count, dtype: float64

Weighted class mass by week bucket (train):
week_bucket
1    0.4949
2    0.3683
3    0.0772
4    0.0190
5    0.0139
6    0.0089
7    0.0179
Name: sample_weight, dtype: float64


 It uses your existing pipeline, but the weighted fit passes weights into the classifier inside the sklearn pipeline.

 ✅ What I want now:

1) baseline balanced accuracy + macro F1
2) weighted balanced accuracy + macro F1
3) per-class recall from the classification report, especially buckets 4, 5, 6, 7

In [8]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from cf_copilot.ml_logic.encoders import preprocess
from cf_copilot.ml_logic.model import initialize_model

# Prepare X/y
X_train_base, y_train_base = preprocess(train_df_w)
X_test_base, y_test_base = preprocess(test_df_w)

# Baseline model
baseline_model = initialize_model()
baseline_model.fit(X_train_base, y_train_base)

baseline_pred = baseline_model.predict(X_test_base)

print("BASELINE")
print("Balanced accuracy:", round(balanced_accuracy_score(y_test_base, baseline_pred), 4))
print("Macro F1:", round(f1_score(y_test_base, baseline_pred, average="macro"), 4))
print(classification_report(y_test_base, baseline_pred))

# Weighted model
weighted_model = initialize_model()
weighted_model.fit(
    X_train_base,
    y_train_base,
    classifier__sample_weight=train_df_w["sample_weight"].values
)

weighted_pred = weighted_model.predict(X_test_base)

print("\nWEIGHTED")
print("Balanced accuracy:", round(balanced_accuracy_score(y_test_base, weighted_pred), 4))
print("Macro F1:", round(f1_score(y_test_base, weighted_pred, average="macro"), 4))
print(classification_report(y_test_base, weighted_pred))


✅ Model (pipeline) initialized
BASELINE
Balanced accuracy: 0.5595
Macro F1: 0.5454
              precision    recall  f1-score   support

           1       0.84      0.88      0.86      9647
           2       0.80      0.70      0.75      7578
           3       0.48      0.52      0.50      2170
           4       0.36      0.44      0.39       783
           5       0.40      0.38      0.39       554
           6       0.35      0.31      0.33       311
           7       0.53      0.68      0.59       768

    accuracy                           0.74     21811
   macro avg       0.54      0.56      0.55     21811
weighted avg       0.75      0.74      0.74     21811

✅ Model (pipeline) initialized

WEIGHTED
Balanced accuracy: 0.5545
Macro F1: 0.5491
              precision    recall  f1-score   support

           1       0.84      0.88      0.86      9647
           2       0.81      0.70      0.75      7578
           3       0.44      0.55      0.49      2170
           4       

In [ ]:
baseline_pred_dist = pd.Series(baseline_pred).value_counts(normalize=True).sort_index().rename("baseline")
weighted_pred_dist = pd.Series(weighted_pred).value_counts(normalize=True).sort_index().rename("weighted")
true_dist = y_test_base.value_counts(normalize=True).sort_index().rename("true")

pred_compare = pd.concat([true_dist, baseline_pred_dist, weighted_pred_dist], axis=1).fillna(0)
display(pred_compare.round(4))
